In [1]:
import pandas as pd
import os
import numpy as np
import librosa
import soundfile as sf
import random
import shutil
import time
import random
from multiprocessing import Pool, cpu_count
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
df_4th = pd.read_csv(r'/content/drive/MyDrive/말동이_project/voice/4차년도.csv', encoding='cp949')
df_5th = pd.read_csv(r'/content/drive/MyDrive/말동이_project/voice/5차년도.csv', encoding='cp949')
df_5th_2nd = pd.read_csv(r'/content/drive/MyDrive/말동이_project/voice/5차년도_2차.csv', encoding='cp949')
dataframes = [df_4th, df_5th, df_5th_2nd]
unite_df = pd.concat(dataframes, ignore_index=True)
print(unite_df.head())

                     wav_id                        발화문     상황    1번 감정  \
0  5e258fd1305bcf3ad153a6a4           어, 청소 니가 대신 해 줘!  anger  Neutral   
1  5e258fe2305bcf3ad153a6a5         둘 다 청소 하기 싫어. 귀찮아.  anger  Neutral   
2  5e258ff5305bcf3ad153a6a6             둘 다 하기 싫어서 화내.  anger    Angry   
3  5e25902f305bcf3ad153a6a9                그럼 방세는 어떡해.  anger  Sadness   
4  5e27f90b5807b852d9e0157b  권태긴줄 알았는데 다른 사람이 생겼나보더라고.    sad  Sadness   

   1번 감정세기    2번 감정  2번 감정세기    3번 감정  3번 감정세기    4번 감정  4번감정세기    5번 감정  \
0        0    Angry        1  Neutral        0  Neutral       0    Angry   
1        0    Angry        1  Neutral        0  Neutral       0    Angry   
2        1    Angry        1  Neutral        0    Angry       1    Angry   
3        1  Sadness        1  Sadness        1  Sadness       1  Sadness   
4        1  Sadness        1  Sadness        1  Sadness       2  Sadness   

   5번 감정세기  나이    성별  
0        1  27  male  
1        1  27  male  
2        1  27  male  
3     

In [3]:
drop_columns = ['1번 감정','1번 감정세기','2번 감정','2번 감정세기','3번 감정','3번 감정세기','4번 감정','4번감정세기','5번 감정','5번 감정세기']
unite_df.drop(columns = drop_columns, axis=1, inplace = True)
unite_df.head()

,wav_id,발화문,상황,나이,성별
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,27,male
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,27,male
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,27,male
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,27,male
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,32,male


In [4]:
base_audio_dir = r'/content/drive/MyDrive/말동이_project/voice/'

# 3개 오디오 파일 합치기
audio_subfolders = [r'/content/drive/MyDrive/말동이_project/voice/4차년도',
                    r'/content/drive/MyDrive/말동이_project/voice/5차년도',
                    r'/content/drive/MyDrive/말동이_project/voice/5차년도_2차']

# 오디오 파일 경로가 맞는지 확인
def find_audio_path(wav_id, base_dir, subfolders):
    for subfolder in subfolders:
        audio_path = os.path.join(base_dir, subfolder, f'{wav_id}.wav')
        if os.path.exists(audio_path):
            return audio_path
    return None

# audio_path column 추가
unite_df['audio_path'] = unite_df['wav_id'].apply(lambda x: find_audio_path(x, base_audio_dir, audio_subfolders))

print(unite_df.head())

                     wav_id                        발화문     상황  나이    성별  \
0  5e258fd1305bcf3ad153a6a4           어, 청소 니가 대신 해 줘!  anger  27  male   
1  5e258fe2305bcf3ad153a6a5         둘 다 청소 하기 싫어. 귀찮아.  anger  27  male   
2  5e258ff5305bcf3ad153a6a6             둘 다 하기 싫어서 화내.  anger  27  male   
3  5e25902f305bcf3ad153a6a9                그럼 방세는 어떡해.  anger  27  male   
4  5e27f90b5807b852d9e0157b  권태긴줄 알았는데 다른 사람이 생겼나보더라고.    sad  32  male   

                                          audio_path  
0  /content/drive/MyDrive/말동이_project/voice/...  
1  /content/drive/MyDrive/말동이_project/voice/...  
2  /content/drive/MyDrive/말동이_project/voice/...  
3  /content/drive/MyDrive/말동이_project/voice/...  
4  /content/drive/MyDrive/말동이_project/voice/...  


In [5]:
# csv파일과 오디오 파일 매칭되지 않는 것들 제거
def check_file_exists(file_path):
  if pd.isna(file_path):
    return False
  return os.path.exists(file_path)

# audio_exists column을 새로 만들어 audio_path가 True인지 False인지 데이터프레임에 추가
unite_df['audio_exists'] = unite_df['audio_path'].apply(check_file_exists)

# audio_exists가 True인 row만 선택해서 df_clean에 copy, False는 drop
df_clean = unite_df[unite_df['audio_exists']].copy()
df_clean.drop(columns=['audio_exists'], inplace=True)

print(f"제거 전 갯수 : {len(unite_df)}")
print(f"제거 후 갯수 : {len(df_clean)}")

제거 전 갯수 : 43991
제거 후 갯수 : 43975


In [6]:
# 상황->감정 변경 / 나이, 성별은 drop
df_clean.rename(columns={'상황' : '감정'}, inplace=True)
df_clean.drop(columns={'나이', '성별'}, inplace=True)
df_clean.head()

,wav_id,발화문,감정,audio_path
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,/content/drive/MyDrive/말동이_project/voice/...
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,/content/drive/MyDrive/말동이_project/voice/...
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,/content/drive/MyDrive/말동이_project/voice/...
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,/content/drive/MyDrive/말동이_project/voice/...
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,/content/drive/MyDrive/말동이_project/voice/...


In [7]:
# 유사한 감정명 통합
emotion_mapping = {'sadness':'sad', 'angry':'anger'}
df_clean['감정'] = df_clean['감정'].replace(emotion_mapping)

# 데이터 균형 파악
emotion_count = df_clean['감정'].value_counts()
print(emotion_count)

감정
sad          13986
anger        11633
disgust       4660
happiness     4548
fear          4131
neutral       3262
surprise      1755
Name: count, dtype: int64


In [8]:
# --- 설정 변수 (유지) ---
TARGET_COUNT = 5000  # 목표 개수 5000 유지
TARGET_SR = 16000    # 16kHz 리샘플링 유지
AUGMENTED_DIR = r'/content/drive/MyDrive/말동이_project/voice/augmented_data_5k'

# 1. 이전 증강 데이터 폴더 초기화 (반복 실행 시 파일 겹침 방지)
if os.path.exists(AUGMENTED_DIR):
    print(f"✅ 이전 증강 데이터 삭제: {AUGMENTED_DIR}")
    shutil.rmtree(AUGMENTED_DIR)

os.makedirs(AUGMENTED_DIR, exist_ok=True)
print(f"새로운 증강 파일 저장 경로: {AUGMENTED_DIR}")

# --- 증강 함수 (5가지 증강 기법 적용) ---
def augment_and_save(args):
    """오디오 파일을 로드하고, 다섯 가지 증강 기법 중 하나를 적용 후 저장하는 함수."""
    row, new_wav_id_base, output_path, emotion = args

    try:
        # 1. 오디오 로드 및 16kHz 리샘플링
        y, sr = librosa.load(row['audio_path'], sr=TARGET_SR)
    except Exception:
        return None

    # 2. 증강 기법 무작위 선택 및 적용
    # 'roll'과 'volume'이 추가되었습니다.
    augmentation_type = random.choice(['noise', 'pitch_shift', 'time_stretch', 'roll', 'volume'])
    y_aug = y.copy()

    # 새로운 ID에 증강 타입을 반영
    new_wav_id = f"{new_wav_id_base}_{augmentation_type}"

    if augmentation_type == 'noise':
        # 📢 노이즈 주입 (±0.015 -> ±0.02로 강도 소폭 확대)
        noise = np.random.uniform(low=-0.02, high=0.02, size=y.shape)
        y_aug = y + noise

    elif augmentation_type == 'pitch_shift':
        # 🎶 Pitch Shift (음높이 변경: -2.0 ~ 2.0 -> -3.0 ~ 3.0 반음으로 확대)
        steps = np.random.uniform(low=-3.0, high=3.0)
        y_aug = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=steps)

    elif augmentation_type == 'time_stretch':
        # ⏱️ Time Stretch (시간 늘이기/줄이기: 0.85 ~ 1.15 비율 유지)
        rate = np.random.uniform(low=0.85, high=1.15)
        y_aug = librosa.effects.time_stretch(y=y, rate=rate)

        # 길이 처리 (원본 길이와 맞추기)
        if len(y_aug) < len(y):
            padding_length = len(y) - len(y_aug)
            y_aug = np.pad(y_aug, (0, padding_length), 'constant')
        elif len(y_aug) > len(y):
            y_aug = y_aug[:len(y)]

    elif augmentation_type == 'roll':
        # 🔄 Time Roll (시간 축 이동)
        # shift: 오디오 길이에 비례한 -100 ~ 100 프레임 이동
        roll_length = np.random.randint(-100, 100)
        y_aug = np.roll(y, roll_length)

    elif augmentation_type == 'volume':
        # 🔊 Volume Change (볼륨 변경: 0.5 ~ 1.5배)
        # 0.5배 (작게) ~ 1.5배 (크게)
        gain = np.random.uniform(low=0.5, high=1.5)
        y_aug = y * gain
        # 클리핑 방지 (최대값 1.0을 넘지 않도록)
        y_aug = np.clip(y_aug, -1.0, 1.0)

    # 3. 증강된 오디오 저장
    try:
        final_output_path = os.path.join(os.path.dirname(output_path), f'{new_wav_id}.wav')
        sf.write(final_output_path, y_aug, TARGET_SR)

        # 새로운 행 데이터 반환
        new_row = row.copy()
        new_row['wav_id'] = new_wav_id
        new_row['audio_path'] = final_output_path
        new_row['augmentation_type'] = augmentation_type # 어떤 증강이 적용됐는지 기록
        return new_row
    except Exception as e:
        # print(f"저장 실패: {e}")
        return None

# --- 데이터 불균형 해소 로직 (변경 없음) ---
balanced_data = []
NUM_CORES = cpu_count()
print(f"사용 가능한 CPU 코어 수: {NUM_CORES}. 멀티프로세싱을 사용하여 증강을 병렬 처리합니다.")

for emotion, group in df_clean.groupby('감정'):
    current_count = len(group)

    if current_count >= TARGET_COUNT:
        # 1. 언더샘플링 (다수 클래스: sad, anger 등)
        sampled_group = group.sample(n=TARGET_COUNT, random_state=42).copy()
        print(f"[{emotion}] 언더샘플링: {current_count} -> {len(sampled_group)}개")

    else:
        # 2. 오버샘플링 (소수 클래스)
        needed_count = TARGET_COUNT - current_count
        sampled_group = group.copy()

        # A) 하이브리드 증강 작업 목록 (부족한 개수 모두 증강으로 채우기)
        augmentation_tasks = []
        indices_to_augment = np.random.choice(group.index, size=needed_count, replace=True)

        for i, original_idx in enumerate(indices_to_augment):
            row = group.loc[original_idx]
            new_wav_id_base = f"{row['wav_id']}_aug_{i}"
            output_sub_dir = os.path.join(AUGMENTED_DIR, emotion)
            os.makedirs(output_sub_dir, exist_ok=True)
            output_path = os.path.join(output_sub_dir, f'{new_wav_id_base}.wav')
            # augment_and_save 함수 내에서 5가지 중 하나가 무작위 적용됩니다.
            augmentation_tasks.append((row, new_wav_id_base, output_path, emotion))


        print(f"[{emotion}] 하이브리드 증강 작업 시작. {len(augmentation_tasks)}개의 파일 생성 필요.")

        # 하이브리드 증강 실행
        with Pool(NUM_CORES) as pool:
            augmentation_results = pool.map(augment_and_save, augmentation_tasks)

        new_augmented_rows = [res for res in augmentation_results if res is not None]

        # 원본 데이터와 증강 데이터 합치기
        if new_augmented_rows:
            sampled_group = pd.concat([sampled_group, pd.DataFrame(new_augmented_rows)], ignore_index=True)

        # 최종 개수 맞추기 (5000개 초과 시 다시 5000개로 언더샘플링 - 증강 실패 대비)
        if len(sampled_group) > TARGET_COUNT:
            sampled_group = sampled_group.sample(n=TARGET_COUNT, random_state=42).copy()

        actual_augmented_count = len(sampled_group) - current_count
        print(f"[{emotion}] 증강 완료: {current_count} -> {len(sampled_group)}개 (총 {actual_augmented_count}개 증강)")

    balanced_data.append(sampled_group)

# 최종 데이터프레임 생성 및 섞기
df_balanced = pd.concat(balanced_data, ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


print("\n--- 데이터 증강 및 균형 맞춤 최종 결과 (TARGET_COUNT=5000) ---")
print(df_balanced['감정'].value_counts().sort_index())
print(f"최종 데이터프레임 총 행 개수: {len(df_balanced)}")

✅ 이전 증강 데이터 삭제: /content/drive/MyDrive/말동이_project/voice/augmented_data_5k
새로운 증강 파일 저장 경로: /content/drive/MyDrive/말동이_project/voice/augmented_data_5k
사용 가능한 CPU 코어 수: 8. 멀티프로세싱을 사용하여 증강을 병렬 처리합니다.
[anger] 언더샘플링: 11633 -> 5000개
[disgust] 하이브리드 증강 작업 시작. 340개의 파일 생성 필요.
[disgust] 증강 완료: 4660 -> 5000개 (총 340개 증강)
[fear] 하이브리드 증강 작업 시작. 869개의 파일 생성 필요.
[fear] 증강 완료: 4131 -> 5000개 (총 869개 증강)
[happiness] 하이브리드 증강 작업 시작. 452개의 파일 생성 필요.
[happiness] 증강 완료: 4548 -> 5000개 (총 452개 증강)
[neutral] 하이브리드 증강 작업 시작. 1738개의 파일 생성 필요.
[neutral] 증강 완료: 3262 -> 5000개 (총 1738개 증강)
[sad] 언더샘플링: 13986 -> 5000개
[surprise] 하이브리드 증강 작업 시작. 3245개의 파일 생성 필요.
[surprise] 증강 완료: 1755 -> 5000개 (총 3245개 증강)

--- 데이터 증강 및 균형 맞춤 최종 결과 (TARGET_COUNT=5000) ---
감정
anger        5000
disgust      5000
fear         5000
happiness    5000
neutral      5000
sad          5000
surprise     5000
Name: count, dtype: int64
최종 데이터프레임 총 행 개수: 35000


In [9]:
N_MFCC = 60 # 기본 MFCC 개수
TARGET_SR = 16000
N_MELS = 128
FRAME_LENGTH = 2048
HOP_LENGTH = 512

# 💡 180차원 롤백 (60 * 3 = 180)
N_MFCC_FINAL = 180 # 최종 특징 차원 (180)

# 특징 벡터를 저장할 경로 (롤백 경로)
FEATURE_DIR = r'/content/drive/MyDrive/말동이_project/voice/mfcc_delta_n180'

# 1. 특징 저장 디렉토리 초기화
if os.path.exists(FEATURE_DIR):
    print(f"✅ 기존 특징 폴더 삭제: {FEATURE_DIR}")
    try:
        shutil.rmtree(FEATURE_DIR)
    except OSError as e:
        print(f"경고: 폴더 삭제 실패. 수동으로 삭제해 주세요: {e}")

os.makedirs(FEATURE_DIR, exist_ok=True)
print(f"새로운 MFCC 특징 파일 저장 경로: {FEATURE_DIR} (차원: {N_MFCC_FINAL})")


# --- MFCC 추출 함수 (Prosody 제거, 180차원만) ---
def extract_mfcc_with_delta(audio_path, target_sr, n_mfcc, frame_length, hop_length):
    try:
        y, sr = librosa.load(audio_path, sr=target_sr)
        # 기본 MFCC 추출
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=frame_length, hop_length=hop_length, n_mels=N_MELS)

        # 1차 미분 (Delta) 특징 추가
        mfcc_delta = librosa.feature.delta(mfcc)

        # 2차 미분 (Delta-Delta) 특징 추가
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

        # 3가지 특징을 수직으로 결합 (운율 특징 없음)
        combined_features = np.vstack([mfcc, mfcc_delta, mfcc_delta2])

        # 정규화
        combined_features = (combined_features - np.mean(combined_features)) / np.std(combined_features)

        return combined_features.T # (time_steps, 180) 형태로 전치하여 반환
    except Exception as e:
        return None

# --- 멀티프로세싱을 위한 래퍼 함수 (함수 호출 롤백) ---
def process_single_row(row):
    """MFCC+Delta 특징을 추출하고 NumPy (.npy) 파일로 저장합니다."""
    output_filename = f"{row['wav_id']}.npy"
    output_path = os.path.join(FEATURE_DIR, output_filename)

    # 초기화 후 재실행 시 건너뛰기 로직
    if os.path.exists(output_path):
        return {'wav_id': row['wav_id'], 'mfcc_path': output_path, 'emotion': row['감정'], 'status': 'Skipped'}

    # 💡 extract_mfcc_with_prosody -> extract_mfcc_with_delta로 롤백
    mfcc_data = extract_mfcc_with_delta(
        audio_path=row['audio_path'],
        target_sr=TARGET_SR,
        n_mfcc=N_MFCC,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH
    )

    if mfcc_data is not None:
        np.save(output_path, mfcc_data)
        return {'wav_id': row['wav_id'], 'mfcc_path': output_path, 'emotion': row['감정'], 'status': 'Extracted'}
    else:
        return {'wav_id': row['wav_id'], 'mfcc_path': None, 'emotion': row['감정'], 'status': 'Failed'}
# --- 멀티프로세싱 실행 ---
start_time = time.time()
print(f"\n--- MFCC 특징 추출 시작 (N_MFCC={N_MFCC_FINAL}, {len(df_balanced)}개 파일) ---")

NUM_CORES = cpu_count()
rows_list = df_balanced.to_dict('records')

with Pool(NUM_CORES) as pool:
    results = list(pool.imap_unordered(process_single_row, rows_list))

mfcc_results_df = pd.DataFrame(results)
df_final = mfcc_results_df[mfcc_results_df['status'] != 'Failed'].copy()
df_final.drop(columns=['status'], inplace=True)

end_time = time.time()
total_time = end_time - start_time
print("\n--- MFCC 특징 추출 완료 ---")
print(f"총 소요 시간: {total_time:.2f}초 ({total_time/60:.2f}분)")
print(f"성공적으로 추출된 특징 개수: {len(df_final)}")
print(f"최종 데이터프레임 (df_final) head:\n{df_final.head()}")

✅ 기존 특징 폴더 삭제: /content/drive/MyDrive/말동이_project/voice/mfcc_delta_n180
새로운 MFCC 특징 파일 저장 경로: /content/drive/MyDrive/말동이_project/voice/mfcc_delta_n180 (차원: 180)

--- MFCC 특징 추출 시작 (N_MFCC=180, 35000개 파일) ---

--- MFCC 특징 추출 완료 ---
총 소요 시간: 2637.62초 (43.96분)
성공적으로 추출된 특징 개수: 35000
최종 데이터프레임 (df_final) head:
                                         wav_id  \
0  5efe1ce3704f492ee1253dcf_aug_44_time_stretch   
1                      5f9226e9111dfd48d40ff126   
2         5e37e6907995ef170fc0f4e0_aug_172_roll   
3                      5f6d91e6f8fac448cc0a600c   
4                      5e3639f28661d6073410fd52   

                                           mfcc_path    emotion  
0  /content/drive/MyDrive/말동이_project/voice/mfcc_...    disgust  
1  /content/drive/MyDrive/말동이_project/voice/mfcc_...  happiness  
2  /content/drive/MyDrive/말동이_project/voice/mfcc_...       fear  
3  /content/drive/MyDrive/말동이_project/voice/mfcc_...    disgust  
4  /content/drive/MyDrive/말동이_project/voice/mfcc_...   

In [10]:
# --- A. 라벨링 및 클래스 설정 (순차적 0~6) ---
# 모델 학습에 사용할 7개 감정의 임시 순차적 레이블
EMOTION_TO_SEQ = {
    'happiness': 0, 'anger': 1, 'sad': 2, 'neutral': 3,
    'disgust': 4, 'fear': 5, 'surprise': 6
}
# 최종적으로 모델이 출력할 클래스 개수는 7개입니다.
NUM_CLASSES = 7

# 최종 목표로 변환하기 위한 역매핑 테이블 (평가 시 사용)
# 임시 순차적 레이블 (0~6) -> 최종 목표 레이블 (0, 2, 5, 6, 7, 8, 9)
FINAL_MAP = {
    0: 0, 1: 2, 2: 5, 3: 6, 4: 7, 5: 8, 6: 9
}


# 1. df_final에 순차적 레이블 적용
df_final['label'] = df_final['emotion'].map(EMOTION_TO_SEQ)

# 2. 특징 경로와 레이블 분리
X = df_final['mfcc_path']
y = df_final['label']

# 3. 훈련/검증/테스트 분리 (stratify로 클래스 비율 유지)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"✅ 학습 클래스 개수: {NUM_CLASSES}")


# --- B. 특징 로드 및 패딩 ---
FEATURE_DIM = 202 # 실제 특징 차원을 반영

# 1. 병렬 로딩을 위한 래퍼 함수
def load_single_feature(path):
    """단일 NPY 파일을 로드합니다. (멀티프로세싱 Pool에서 사용)"""
    try:
        mfcc = np.load(path)
        return mfcc
    except Exception:
        # 로드 실패 시 빈 데이터 반환 (패딩이 처리할 수 있도록)
        return np.zeros((1, FEATURE_DIM), dtype=np.float32)


def parallel_load_and_pad(file_paths_series, max_len=None):
    """파일 로딩을 병렬로 수행하고, 결과를 합쳐 패딩합니다."""

    # 1. 파일 로딩 (멀티프로세싱)
    print(f"[{time.strftime('%H:%M:%S')}] 🚀 특징 파일 로드 시작 (병렬 처리)")
    with Pool(cpu_count()) as pool:
        # file_paths_series는 Pandas Series 형태이므로 .values를 사용합니다.
        features = list(pool.map(load_single_feature, file_paths_series.values))
    print(f"[{time.strftime('%H:%M:%S')}] ✅ 특징 파일 로드 완료. (RAM 부하 시작)")

    lengths = [f.shape[0] for f in features]

    # 2. MAX_TIMESTEPS 결정
    if max_len is None:
        max_len = int(np.percentile(lengths, 90))
        MAX_TIMESTEPS = min(max_len, 350)
    else:
        MAX_TIMESTEPS = max_len

    # 3. 패딩 (순차 처리)
    print(f"[{time.strftime('%H:%M:%S')}] 🔄 패딩 시작: {MAX_TIMESTEPS} Timesteps")
    padded_sequences = pad_sequences(
        features,
        maxlen=MAX_TIMESTEPS,
        dtype='float32',
        padding='post',
        truncating='post',
        value=0.0
    )
    return padded_sequences, MAX_TIMESTEPS


# 1. 훈련 데이터 로드 및 MAX_TIMESTEPS 결정
X_train_padded, MAX_TIMESTEPS = parallel_load_and_pad(X_train, max_len=None)
# 2. 결정된 MAX_TIMESTEPS로 검증/테스트 데이터 패딩
X_val_padded, _ = parallel_load_and_pad(X_val, MAX_TIMESTEPS)
X_test_padded, _ = parallel_load_and_pad(X_test, MAX_TIMESTEPS)

# 3. 최종 OHE 및 입력 형태 정의
y_train_ohe = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_ohe = to_categorical(y_val, num_classes=NUM_CLASSES)
y_test_ohe = to_categorical(y_test, num_classes=NUM_CLASSES)

N_MFCC_FINAL = X_train_padded.shape[2]
INPUT_SHAPE = (MAX_TIMESTEPS, N_MFCC_FINAL)

print(f"모델 입력 형태: {INPUT_SHAPE}")
print(f"OHE 레이블 형태 (y_train): {y_train_ohe.shape}")

✅ 학습 클래스 개수: 7
[18:34:27] 🚀 특징 파일 로드 시작 (병렬 처리)
[18:36:04] ✅ 특징 파일 로드 완료. (RAM 부하 시작)
[18:36:04] 🔄 패딩 시작: 287 Timesteps
[18:36:06] 🚀 특징 파일 로드 시작 (병렬 처리)
[18:36:35] ✅ 특징 파일 로드 완료. (RAM 부하 시작)
[18:36:35] 🔄 패딩 시작: 287 Timesteps
[18:36:36] 🚀 특징 파일 로드 시작 (병렬 처리)
[18:37:04] ✅ 특징 파일 로드 완료. (RAM 부하 시작)
[18:37:04] 🔄 패딩 시작: 287 Timesteps
모델 입력 형태: (287, 180)
OHE 레이블 형태 (y_train): (21000, 7)


In [11]:
DATASET_DIR = r'/content/drive/MyDrive/말동이_project/voice/final_training_data_n180'
# 저장할 변수 목록과 파일 이름 매핑
data_to_save = {
    'X_train_padded.npy': X_train_padded,
    'X_val_padded.npy': X_val_padded,
    'X_test_padded.npy': X_test_padded,
    'y_train_ohe.npy': y_train_ohe,
    'y_val_ohe.npy': y_val_ohe,
    'y_test_ohe.npy': y_test_ohe
}

# 1. 저장 디렉토리 생성
os.makedirs(DATASET_DIR, exist_ok=True)

# 2. 데이터 저장
print(f"--- 최종 학습 데이터 저장 시작: {DATASET_DIR} ---")
for filename, data_array in data_to_save.items():
    save_path = os.path.join(DATASET_DIR, filename)
    np.save(save_path, data_array)
    print(f"✅ {filename} 저장 완료. Shape: {data_array.shape}")

print("\n🎉 모든 학습 데이터가 Google Drive에 성공적으로 저장되었습니다.")

--- 최종 학습 데이터 저장 시작: /content/drive/MyDrive/말동이_project/voice/final_training_data_n180 ---
✅ X_train_padded.npy 저장 완료. Shape: (21000, 287, 180)
✅ X_val_padded.npy 저장 완료. Shape: (7000, 287, 180)
✅ X_test_padded.npy 저장 완료. Shape: (7000, 287, 180)
✅ y_train_ohe.npy 저장 완료. Shape: (21000, 7)
✅ y_val_ohe.npy 저장 완료. Shape: (7000, 7)
✅ y_test_ohe.npy 저장 완료. Shape: (7000, 7)

🎉 모든 학습 데이터가 Google Drive에 성공적으로 저장되었습니다.
